<a href="https://colab.research.google.com/github/ArshCypherZ/college-buildings-classifier/blob/dev/college_buildings_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!unzip /content/dataset.zip

Archive:  /content/dataset.zip
  inflating: paramedical block/20250104_160221.jpg  
  inflating: paramedical block/20250104_160241.jpg  
  inflating: paramedical block/20250104_160243.jpg  
  inflating: paramedical block/20250104_160244.jpg  
  inflating: paramedical block/20250104_160246.jpg  
  inflating: paramedical block/20250104_160247.jpg  
  inflating: paramedical block/20250104_160249.jpg  
  inflating: paramedical block/20250104_160302.jpg  
  inflating: paramedical block/20250104_160304.jpg  
  inflating: paramedical block/20250104_160305.jpg  
  inflating: paramedical block/20250104_160307.jpg  
  inflating: paramedical block/20250104_160308.jpg  
  inflating: paramedical block/20250104_160309.jpg  
  inflating: paramedical block/20250104_160313.jpg  
  inflating: paramedical block/20250104_160314.jpg  
  inflating: paramedical block/20250104_160315.jpg  
  inflating: paramedical block/20250104_160316.jpg  
  inflating: paramedical block/20250104_160331.jpg  
  inflating: pa

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Function to load dataset
def load_dataset(dataset_path, img_size=(64, 64)):
    images = []
    labels = []

    for label_name in os.listdir(dataset_path):
        label_path = os.path.join(dataset_path, label_name)
        if os.path.isdir(label_path):
            for img_name in os.listdir(label_path):
                img_path = os.path.join(label_path, img_name)
                img = cv2.imread(img_path)
                if img is not None:
                    img = cv2.resize(img, img_size)  # Resize image
                    images.append(img.flatten())  # Flatten the image
                    labels.append(label_name)

    return np.array(images), np.array(labels)

# Load the dataset
dataset_path = "/content/dataset"
img_size = (64, 64)  # Size to resize images
images, labels = load_dataset(dataset_path, img_size)

# Encode labels
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)

# Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(images, encoded_labels, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train SVM classifier
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train, y_train)

# Evaluate the model
y_pred = svm_model.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Function to predict the image name
def predict_image_name(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Image not found or invalid image path.")
    img = cv2.resize(img, img_size).flatten()  # Resize and flatten the image
    img = scaler.transform([img])  # Standardize the image
    pred_label = svm_model.predict(img)
    return label_encoder.inverse_transform(pred_label)[0]

# Predict example image
example_image_path = "/content/silver_test.jpg"
predicted_name = predict_image_name(example_image_path)
print(f"Predicted name: {predicted_name}")


                   precision    recall  f1-score   support

    csit building       0.75      1.00      0.86         6
paramedical block       1.00      0.50      0.67         4
    silver jublee       1.00      1.00      1.00         4

         accuracy                           0.86        14
        macro avg       0.92      0.83      0.84        14
     weighted avg       0.89      0.86      0.84        14

Predicted name: silver jublee
